# AI陷阱记录

> 本文件记录本章 C 组在核验中发现的候选陷阱、已确认错误、已排除候选与教学构造。状态字段按实际核验结果填写；未检查不能记成未发现，AI 提示的候选问题不能直接记成已确认错误。

状态：已记录 4 条候选，其中 2 条已确认、1 条不成立、1 条条件不足；另有 1 条教学构造单列。

## 检查范围
- 承担本章 C 角色的实际小组编号、轮次、人工核验者：G1，第 1 轮
- 章节：第 01 章 误差的代价
- 被审版本：A1-G20 正文（更新版），A2-G22 正文；
- 检查文件与单元：A1 §1.1–§1.4 正文、表 1.1–1.14、式 (1.1)–(1.47)；A2 1.1–1.3 节及实验表
- 检查方法：人工复核清单逐项核对、手动推导重推主要结论、NumPy/SciPy 脚本复算关键数值、与 Higham/Goldberg/NumPy 文档对照

## 陷阱条目

### 编号：CH01-T001
- **状态**：已确认（编辑口径问题，非数学错误）
- **来源**：人工核对两稿符号（非 AI 回复）
- **AI工具、模型与日期**：不适用（人工发现）；相关 AI 对话见 `C1组-AI对话记录.md` 第 1 条
- **提示词与AI原话**：不适用；AI 仅确认 `finfo.eps` 指 $2^{1-p}$，未指出教材符号混用
- **错误主张（候选）**：读者可能把“机器精度”等同于 $u=2^{-p}$，而 NumPy `eps` 返回 $2^{1-p}$；两稿符号不同但未在同一处并列说明
- **隐含假设**：读者已知不同资料对“机器精度”命名不同
- **最小反例**：`np.finfo(np.float64).eps == 2**-52` 为 `True`，而 $2^{-53}$ 是 $u$；若把 `eps` 当作单次舍入相对误差界，会高估一倍
- **预期与实测**：预期教材首次出现处明确二者关系；实测 A1 式 (1.5)–(1.6) 已写 $u=2^{-p}$、$\epsilon_{\mathrm{mach}}=2^{1-p}=2u$，A2 也分列 $\varepsilon$ 与 $u$，但两稿符号名不同
- **验证代码来源与人工核验**：本组手写 `np.finfo` 检查；人工对照 A1 参考文献 [5]
- **原因分析**：非数学错误，属跨稿符号统一问题；单稿内部自洽
- **影响与严重程度**：低—中；不影响结论，但影响读者跨资料阅读
- **修复方式**：综合稿统一写 $u=2^{-p}$、$\epsilon_{\mathrm{mach}}=2^{1-p}$，首次出现处注明 `finfo.eps` 指后者
- **回归验证**：待综合稿定稿后检查式 (1.5)–(1.6) 与 1.4 节低精度讨论的符号一致性
- **对教材的修改**：建议在 §1.1 加一句“本章 $u$ 与 $\epsilon_{\mathrm{mach}}$ 的分工”；不改变现有公式

### 编号：CH01-T002
- **状态**：已确认（表述风险，非计算错误）
- **来源**：人工核对 A1 §1.2 与 A2 1.2 节
- **AI工具、模型与日期**：不适用（人工发现）
- **提示词与AI原话**：不适用
- **错误主张（候选）**：读者可能把“步长越小越准”或“代数等价即可靠”当作普遍规则
- **隐含假设**：忽略截断与舍入的竞争、忽略浮点实现
- **最小反例**：A1 表 1.6 / A2 表 1.2：$h=10^{-16}$ 时 $1+h$ 舍回 1，前向差分输出 0，绝对误差约 $e$；而 $h=10^{-8}$ 时误差约 $6.6\times10^{-9}$
- **预期与实测**：预期教材明确“步长需平衡”；实测两稿均有此表述，且 A1 图 1.2、A2 图 01-2 展示回升
- **验证代码来源与人工核验**：本组复算 $h=10^{-8}$ 与 $10^{-16}$ 的差分，确认输出 0 与约 $6.6\times10^{-9}$
- **原因分析**：非教材错误；属读者易误读点，教材已用实验与文字覆盖
- **影响与严重程度**：低；作为教学提示保留
- **修复方式**：综合稿保留“步长平衡”段与图，不删实验
- **回归验证**：检查图注是否写明“观测最小点依赖网格与实现”
- **对教材的修改**：建议在图 1.2 图注加“最小点随精度、函数与网格变化”

### 编号：CH01-T003
- **状态**：不成立（候选被排除）
- **来源**：AI 回复候选（见 `C1组-AI对话记录.md` 第 3 条）
- **AI工具、模型与日期**：__；2026-11-__
- **提示词与AI原话**：“反向递推总比正向递推更准吗？”AI 回复“不能一概而论，若末值误差很大仍可能不准”
- **错误主张（候选）**：“反向递推总更准”这一说法若被写入教材即为错误
- **隐含假设**：末值精度足够、新增舍入不放大
- **最小反例**：人工构造 $N=9$、$\widehat I_9=0.5$（远超上界 0.1），反推 $I_0$ 误差约 $1.38\times10^{-6}$，仍大于 $10^{-7}$
- **预期与实测**：预期“反向更准”需末值在合理范围；实测 A1 例 1.5 用 $0.0684$ 落在 $(0.0368,0.1)$ 内，故有效
- **验证代码来源与人工核验**：本组手写反推脚本；人工核对 A1 式 (1.32) 范围
- **原因分析**：A1 正文已写“末值虽然粗糙，却足以使低阶积分明显改善；给定的末端 $I_9$ 本身不会因此变准确”，并未主张“总更准”。候选主张不成立
- **影响与严重程度**：无；作为排除记录
- **修复方式**：无需修改；综合稿保留 A1 的限定语
- **回归验证**：不适用
- **对教材的修改**：无

### 编号：CH01-T004
- **状态**：条件不足（暂不能判断）
- **来源**：AI 回复中的定性提示（见 `C1组-AI对话记录.md` 第 3 条）
- **AI工具、模型与日期**：__；2026-11-__
- **提示词与AI原话**：AI 称“若递推方向改变后新增舍入在每一步被放大，也可能抵消缩小效果”
- **错误主张（候选）**：反向递推的新增舍入可能在某些参数下主导误差
- **隐含假设**：每步舍入被 $1/n$ 缩小，但局部舍入量可能随 $n$ 增大
- **最小反例**：未构造出具体反例；A1 正文只分析“忽略新增舍入”的传播，未给新增舍入的完整界
- **预期与实测**：预期需给出 $\rho_n$ 的量级与累积界；实测 A1 未展开，A2 未涉及
- **验证代码来源与人工核验**：本组尝试用随机 $\rho_n$ 模拟，未得到稳定反例；标为条件不足
- **原因分析**：缺少对新增舍入的定量模型；不宜据此修改教材
- **影响与严重程度**：低；作为开放问题记录
- **修复方式**：不写入教材；可在习题中作为探究题
- **回归验证**：待有定量模型后复核
- **对教材的修改**：无

### 编号：CH01-T005（教学构造，非 AI 回复）
- **状态**：教学构造（用于课堂，不作为错误记录）
- **来源**：人工教学构造
- **AI工具、模型与日期**：不适用
- **提示词与AI原话**：不适用
- **错误主张（构造）**：学生常写“$\mathrm{fl}(a+b)=a+b$ 对所有 $a,b$ 成立”或“$0.1+0.2=0.3$ 在 float64 中精确”
- **隐含假设**：忽略舍入模型
- **最小反例**：`0.1+0.2 == 0.3` 在 Python 中为 `False`；`math.sqrt(10**16+1)-math.sqrt(10**16)==0`
- **预期与实测**：预期学生先预测再运行；实测与 A1 表 1.3 一致
- **验证代码来源与人工核验**：本组手写；输出与 A1 表 1.3 对照
- **原因分析**：教学构造，用于引出舍入模型
- **影响与严重程度**：不适用
- **修复方式**：不适用
- **回归验证**：不适用
- **对教材的修改**：建议保留 A1 的“先预测再运行”设计

## 未覆盖范围与局限
- 未逐行审计 A1 全部 47 个编号公式的每一步代数；重点覆盖标为主要结论的式子。
- 未独立复现 A1 图 1.1、1.2 的全部绘图代码；仅核对图注与表中数值。
- 未验证 A2 1.4 节（A2 未展开），故不对其作结论。
- AI 对话记录仅涵盖本组实际发起的 4 条；未使用 AI 生成最终教材文字。